<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/JARVIS_ML_LightGBM_GPUvsCPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## This notebook shows how GPU can accelerate model training with respect to CPU using JARVIS-ML CFID dataset .

## Install LightGBM with GPU
## Make sure you have enabled GPU using: " Edit>Notebook Settings>GPU " from the dropdown menu up

In [ ]:
!sudo apt install nvidia-driver-460 nvidia-cuda-toolkit clinfo
!apt-get update --fix-missing
!pip install lightgbm --install-option=--gpu --install-option="--opencl-include-dir=/usr/local/cuda/include/" --install-option="--opencl-library=/usr/local/cuda/lib64/libOpenCL.so"


In [ ]:
# ! git clone --recursive https://github.com/Microsoft/LightGBM
# ! cd LightGBM && rm -rf build && mkdir build && cd build && cmake -DUSE_GPU=1 ../../LightGBM && make -j4 && cd ../python-package && python3 setup.py install --precompile --gpu;

##GPU information

In [ ]:
!nvidia-smi

##Install jarvis-tools

In [ ]:
!pip install jarvis-tools

##Get example data from JARVIS

In [ ]:
from jarvis.ai.pkgs.utils import get_ml_data
from jarvis.ai.pkgs.utils import regr_scores
X,y,jid=get_ml_data()
#Formation energy for 3D materials, you can choose other properties/dataset as
# Take 1000 samples only for tutorial purposes
X=X[0:1000]
y=y[0:1000]
jid=jid[0:1000]

##Formation energy model training with GPU
### Other hyperparameters for materials properties are available at https://github.com/usnistgov/jarvis/tree/master/jarvis/ai/pkgs/lgbm

In [ ]:
!mkdir -p /etc/OpenCL/vendors && echo "libnvidia-opencl.so.1" > /etc/OpenCL/vendors/nvidia.icd


In [ ]:
%%capture
import lightgbm as lgb
from sklearn.model_selection import train_test_split
lgbm = lgb.LGBMRegressor(device= 'gpu',n_estimators= 1170,learning_rate= 0.15375236057119931,num_leaves= 273)

X_train, X_test, y_train, y_test, jid_train, jid_test = train_test_split(X, y, jid, random_state=1, test_size=.1)
lgbm.fit(X_train,y_train)
pred = lgbm.predict(X_test)
reg_sc = regr_scores(y_test, pred)


In [ ]:
print (reg_sc['mae'])

##Accuracy of the model can be further improved by increasing number of trees, decreasing learning rate, and increasing number of leaves. However, it requires more training time and the model may overfit. You can skip this step, if you just want to see the speed up

In [ ]:
%%capture
import time
import lightgbm as lgb
from sklearn.model_selection import train_test_split
t1=time.time()
lgbm = lgb.LGBMRegressor(device= 'gpu',n_estimators= 1170,learning_rate= 0.05375236057119931,num_leaves= 573)

X_train, X_test, y_train, y_test, jid_train, jid_test = train_test_split(X, y, jid, random_state=1, test_size=.1)
lgbm.fit(X_train,y_train)
pred = lgbm.predict(X_test)
reg_sc = regr_scores(y_test, pred)
t2=time.time()
total_time=t2-t1



In [ ]:
print (reg_sc['mae'],total_time)

## Formation energy model training with CPU

In [ ]:
%%capture
import lightgbm as lgb
from jarvis.ai.pkgs.utils import regr_scores
from sklearn.model_selection import train_test_split
t1=time.time()

lgbm = lgb.LGBMRegressor(n_estimators= 1170,learning_rate= 0.15375236057119931,num_leaves= 273, device= 'cpu')
X_train, X_test, y_train, y_test, jid_train, jid_test = train_test_split(X, y, y, random_state=1, test_size=.1)
lgbm.fit(X_train,y_train)
pred = lgbm.predict(X_test)
reg_sc = regr_scores(y_test, pred)
print (reg_sc['mae'])
t2=time.time()
total_time=t2-t1

In [ ]:
print (reg_sc['mae'],total_time)